In [1]:
%%writefile sintactico_ast.py
class NodoAST:
    def traducirPy(self): 
        raise NotImplementedError('Metodo traducirPy() no implementado en este Nodo')
    def traducirCPP(self): 
        raise NotImplementedError('Metodo traducirCPP() no implementado en este Nodo')
    def traducirGo(self): 
        raise NotImplementedError('Metodo traducirGO() no implementado en este Nodo')

    def generarCodigo():
        #Traduccion de C++ a ASSEMBLER
        raise NotImplementedError('Metodo generarCodigo() no implementado en este Nodo')

#----------------------Nodo Programa---------------------------------------
class NodoPrograma(NodoAST):
    #Nodo que representa a un programa completo
    def __init__(self, funcion):
        self.variables = []
        self.funcion = funcion

    def generarCodigo(self):
        codigo = ["Section .text","global _start"]
        data = ["section .bss"]
        # Generar codigo de toas las funciones
        for funcion in self.funciones:
            codigo.append(funcion.generarCodigo())
            self.variables.append((funcion.cuerpo[0].tipo[1], funcion.cuerpo[0].nombre[1]))
            if len(funcion.parametros) > 0:
                for parametro in funcion.parametros:
                    self.variables.append((parametro.tipo[1], parametro.nombre[1]))
        

        # Generar el punto de entrada del programa
        codigo.append("_start:")
        codigo.append(self.main.generarCodigio())
        # Finalizacion del Programa
        codigo.append("    mov eax, 1    ; syscall exit")
        codigo.append("    xor ebx, ebx  ; Codigo de Salida 0")
        codigo.append("    int 0x80")

        # Seccion de reserva de memoria para las variables
        for variable in self.variables:
            if variable[0] == 'int':
                data.append(f'    {variable[1]}:    resd 1')
        codigo = '\n'.join(codigo)
        return '\n'.join(data) + '\n' + codigo
            

    def traducirPy(self):
        return "\n\n".join(f.traducirPy() for f in self.funcion)
        
    def traducirCPP(self):
        return "\n\n".join(f.traducirCPP() for f in self.funcion)
        
    def traducirGo(self):
        return "\n\n".join(f.traducirGo() for f in self.funcion)
        
class NodoLlamada(NodoAST):
    def __init__(self, nombre, argumentos):
        self.nombre = nombre
        self.argumentos = argumentos
        
    def traducirPy(self):
        args = ", ".join(a.traducirPy() for a in self.argumentos)
        return f"{self.nombre[1]}({args})"
        
    def traducirCPP(self):
        args = ", ".join(a.traducirCPP() for a in self.argumentos)
        return f"{self.nombre[1]}({args})"
        
    def traducirGo(self):
        args = ", ".join(a.traducirGo() for a in self.argumentos)
        return f"{self.nombre[1]}({args})"
    
class NodoString(NodoAST):
    def __init__(self, valor): self.valor = valor
    def traducirPy(self): return self.valor[1]
    def traducirCPP(self): return self.valor[1]
    def traducirGo(self): return self.valor[1]

class NodoPrint(NodoAST):
    def __init__(self, expresiones): self.expresiones = expresiones
    def traducirPy(self):
        args = ", ".join(e.traducirPy() for e in self.expresiones)
        return f"print({args})"
    def traducirCPP(self):
        args = " << ".join(e.traducirCPP() for e in self.expresiones)
        return f"std::cout << {args} << std::endl"
    def traducirGo(self):
        args = ", ".join(e.traducirGo() for e in self.expresiones)
        return f"fmt.Println({args})"

class NodoFuncion(NodoAST):
    def __init__(self, tipo, nombre, parametros, cuerpo):
        self.tipo = tipo; self.nombre = nombre; self.parametros = parametros; self.cuerpo = cuerpo

    def generarCodigo(self):
        codigo = f'{self.nombre[1]}:\n'
        if len(self.parametros) > 0:
            # Aqui guardamos en pila el registro ax que usaremos
            for parametro in self.parametros:
                codigo += '\n    pop    eax'
                codigo += f'\n    mov [{parametro.nombre[1]}], eax'
        codigo += '\n'.join(c.generarCodigo() for c in self.cuerpo)
        codigo += '\n    ret'
        codigo += '\n'
        return codigo
                

    def traducirPy(self):
        params = ', '.join(p.traducirPy() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirPy() for c in self.cuerpo)
        return f"def {self.nombre[1]}({params}):\n    {cuerpo}"

    def traducirCPP(self):
        tipo_ret = self.tipo[1]; nombre = self.nombre[1]
        params = ', '.join(p.traducirCPP() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirCPP() for c in self.cuerpo)
        return f"{tipo_ret} {nombre}({params}) {{\n    {cuerpo};\n}}"
    
    def traducirGo(self):
        tipo_go = self.tipo[1]
        if tipo_go == 'float' or tipo_go == 'double': tipo_go = 'float64'
        if tipo_go == 'void': tipo_go = '' # En Go si es void no se pone nada
        
        nombre = self.nombre[1]
        params = ', '.join(p.traducirGo() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirGo() for c in self.cuerpo)
        
        retorno_str = f" {tipo_go}" if tipo_go else ""
        return f"func {nombre}({params}){retorno_str} {{\n    {cuerpo}\n}}"

class NodoParametros(NodoAST):
    def __init__(self, tipo, nombre): 
        self.tipo = tipo 
        self.nombre = nombre
    def traducirPy(self): 
        return self.nombre[1]
        
    def traducirCPP(self): 
        return self.nombre[1]
        
    def traducirGo(self):
        tipo_go = self.tipo[1]
        if tipo_go == 'float' or tipo_go == 'double': tipo_go = 'float64'
        return f"{self.nombre[1]} {tipo_go}"

class NodoAsignacion(NodoAST):
    def __init__(self, tipo, nombre, expresion): 
        self.tipo = tipo 
        self.nombre = nombre 
        self.expresion = expresion

    def generarCodigo(self):
        codigo = self.expresion.generarCodigo()
        codigo += f'\n    mov [{self.nombre[1]}], eax'
        return codigo
    
    def traducirPy(self): 
        return f'{self.nombre[1]} = {self.expresion.traducirPy()}'
    def traducirCPP(self): 
        return f'{self.tipo[1]} {self.nombre[1]} = {self.expresion.traducirCPP()}'
    def traducirGo(self):
        tipo_go = self.tipo[1]
        if tipo_go == 'float' or tipo_go == 'double': tipo_go = 'float64'
        return f"var {self.nombre[1]} {tipo_go} = {self.expresion.traducirGo()}"

class NodoOperacion(NodoAST):
    def __init__(self, izquierda, operador, derecha): 
        self.izquierda = izquierda 
        self.operador = operador 
        self.derecha = derecha
        
    def traducirPy(self): 
        return f'{self.izquierda.traducirPy()} {self.operador[1]} {self.derecha.traducirPy()}'
        
    def traducirCPP(self): 
        return f'{self.izquierda.traducirCPP()} {self.operador[1]} {self.derecha.traducirCPP()}'
        
    def traducirGo(self): 
        return f'{self.izquierda.traducirGo()} {self.operador[1]} {self.derecha.traducirGo()}'

    def generarCodigo(self):
        codigo = []
        codigo.append(self.izquierda.generarCodigo())
        codigo.append('    push    eax')
        codigo.append(self.derecha.generarCodigo())
        codigo.append('    mov    ebx, eax')
        codigo.append('    pop    eax')
        if self.operador[1] == '+':
            codigo.append('    add    eax, ebx')
        return '\n'.join(codigo)

class NodoRetorno(NodoAST):
    def __init__(self, expresion): 
        self.expresion = expresion
        
    def traducirPy(self): 
        return f'return {self.expresion.traducirPy()}'
        
    def traducirCPP(self): 
        return f'return {self.expresion.traducirCPP()}'
        
    def traducirGo(self):
        return f'return {self.expresion.traducirGo()}'

    def generarCodigo(self):
        return self.expresion.generarCodigo()

class NodoIdentificador(NodoAST):
    def __init__(self, nombre): 
        self.nombre = nombre
    
    def traducirPy(self): 
        return self.nombre[1]
    
    def traducirCPP(self): 
        return self.nombre[1]
    
    def traducirGo(self): 
        return self.nombre[1]

    def generarCodigo(self):
        return f'\n    mov eax, [{self.nombre[1]}]'

class NodoNumero(NodoAST):
    def __init__(self, valor): 
        self.valor = valor
    
    def traducirPy(self): 
        return str(self.valor[1])
    
    def traducirCPP(self): 
        return str(self.valor[1])
    
    def traducirGo(self): 
        return str(self.valor[1])

    def generarCodigo(self):
        eturn f'\n    mov eax, [{self.valor[1]}]'
        

#----------------Analisador Sintactico------------------------------------

class Parser:
    def __init__(self, tokens): 
        self.tokens = tokens 
        self.pos = 0
        
    def obtener_token_actual(self): 
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    
    def coincidir(self, tipo_esperado):
        token = self.obtener_token_actual()
        if token and token[0] == tipo_esperado: 
            self.pos += 1; return token
        else: 
            raise SyntaxError(f'Error sintactico: se esperaba {tipo_esperado}, pero se encontro: {token}')

    def parsear(self): 
        funciones = []
        # Leer todas las funciones hasta que se acaben los tokens
        while self.obtener_token_actual() is not None:
            funciones.append(self.funcion())
        return NodoPrograma(funciones)

    def funcion(self):
        tipo = self.coincidir('KEYWORD') # Tipo de retorno (ej. int)
        nombre = self.coincidir('IDENTIFIER') # Nombre de la funcion
        self.coincidir('DELIMITER') # Se espera un (
        if nombre == 'main':
            parametros = []
        else:
            parametros = self.parametros() # Regla para los parametros
        self.coincidir('DELIMITER') # Se espera un )
        self.coincidir('DELIMITER') # Se espera un {
        cuerpo = self.cuerpo(); # Regla para el cuerpo de la funcion
        self.coincidir('DELIMITER') # Se espera un }
        return NodoFuncion(tipo, nombre, parametros, cuerpo)

    def parametros(self):
        lista = []
        token = self.obtener_token_actual()
        if token and token[1] == ')': return lista
        
        tipo = self.coincidir('KEYWORD') 
        nombre = self.coincidir('IDENTIFIER')
        lista.append(NodoParametros(tipo, nombre))
        while self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
            self.coincidir('DELIMITER')
            tipo = self.coincidir('KEYWORD')
            nombre = self.coincidir('IDENTIFIER')
            lista.append(NodoParametros(tipo, nombre))
        return lista

    def cuerpo(self):
        instrucciones = []
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != '}':
            token = self.obtener_token_actual()
            if token[1] == 'return': 
                instrucciones.append(self.retorno())
            elif token[1] == 'Console': 
                instrucciones.append(self.sentencia_imprimir_csharp())
            elif token[1] in ['print', 'printf']: 
                instrucciones.append(self.sentencia_imprimir_cpp())
            else: 
                instrucciones.append(self.asignacion())
        return instrucciones

    def sentencia_imprimir_csharp(self):
        token = self.obtener_token_actual()
        if token[0] == 'IDENTIFIER':
            self.coincidir('IDENTIFIER') 
        else: 
            self.coincidir('KEYWORD')
        self.coincidir('DELIMITER')
        if self.obtener_token_actual()[1] == 'WriteLine': 
            self.pos += 1
        else: 
            raise SyntaxError("Se esperaba WriteLine")
        self.coincidir('DELIMITER'); args = self.lista_argumentos()
        self.coincidir('DELIMITER'); self.coincidir('DELIMITER')
        return NodoPrint(args)

    def sentencia_imprimir_cpp(self):
        self.coincidir('KEYWORD'); self.coincidir('DELIMITER')
        args = self.lista_argumentos(); self.coincidir('DELIMITER'); self.coincidir('DELIMITER')
        return NodoPrint(args)

    def lista_argumentos(self):
        args = []
        if self.obtener_token_actual()[1] != ')':
            sigue = True
            while sigue:
                args.append(self.expresion())
                if self.obtener_token_actual()[1] == ',': self.coincidir('DELIMITER')
                else: sigue = False
        return args

    def asignacion(self):
        tipo = self.coincidir('KEYWORD') 
        nombre = self.coincidir('IDENTIFIER')
        self.coincidir('OPERATOR')
        exp = self.expresion()
        self.coincidir('DELIMITER')
        return NodoAsignacion(tipo, nombre, exp)

    def retorno(self):
        self.coincidir('KEYWORD'); exp = self.expresion(); self.coincidir('DELIMITER')
        return NodoRetorno(exp)

    def expresion(self):
        izq = self.termino()
        while self.obtener_token_actual() and self.obtener_token_actual()[0] == 'OPERATOR':
            op = self.coincidir('OPERATOR'); der = self.termino()
            izq = NodoOperacion(izq, op, der)
        return izq

    def termino(self):
        token = self.obtener_token_actual()
        if token[0] == 'NUMBER': 
            return NodoNumero(self.coincidir('NUMBER'))
        elif token[0] == 'STRING': 
            return NodoString(self.coincidir('STRING'))
        elif token[0] == 'IDENTIFIER': 
            identificador = self.coincidir('IDENTIFIER')
            # Verificar si el siguiente token es un '(' para saber si es una llamada a función
            siguiente = self.obtener_token_actual()
            if siguiente and siguiente[1] == '(':
                self.coincidir('DELIMITER') # Consumir el '('
                args = self.lista_argumentos()
                self.coincidir('DELIMITER') # Consumir el ')'
                return NodoLlamada(identificador, args)
            return NodoIdentificador(identificador)
        raise SyntaxError(f"Token inesperado: {token}")


Overwriting sintactico_ast.py
